# Realised utility of volatility forecasts

This notebook discovers every model folder under `Results` containing a same-named main prediction file, loads realised `QMLE-Trade` volatility from `Data/<ticker>.csv`, and calculates the realised utility (RU) of each forecast model.

It follows Zhang et al. (2024), Section 4.4, Equation (21), using the paper's Sharpe ratio $SR=0.4$ and relative risk aversion $\gamma=2$. Because the repository stores volatility rather than log variance, Equation (21) simplifies to

$$RU_{i,t}^{\theta}=\frac{SR^2}{\gamma}\frac{\sigma_{i,t}}{\widehat{\sigma}_{i,t}^{\theta}}-\frac{SR^2}{2\gamma}\left(\frac{\sigma_{i,t}}{\widehat{\sigma}_{i,t}^{\theta}}\right)^2.$$

The notebook first averages utility across assets on each date, then across dates. All models are evaluated on the same asset-date observations. Results are shown as percent of wealth (`100 × RU`), matching the scale of the paper's tables. Under the default assumptions, a perfect forecast has RU of 4.0.

The paper's transaction-cost measure (RU-TC) needs each asset's full median bid-ask spread over the prior 90 trading days. Those data are not present in this repository, so this notebook reports frictionless RU only and does not invent transaction costs.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display

# User settings
MODELS = ["LSTM_10", "LSTM_21", "LSTM_42", "LSTM_63", "LSTM_VAL", "LSTM_AIC"]  # None: every non-robustness folder
INCLUDE_ROBUSTNESS = False       # include robustness folders (*_refit<N>m, *_aic<N>y, *_resel<N>m, *_seed<N>)
ESTIMATE_TYPE = "QMLE-Trade"
VALUE_COLUMN = "Volatility"
DUPLICATE_POLICY = "drop"        # must match duplicate_policy in the model notebooks
VALUES_ARE_VOLATILITY = True
SHARPE_RATIO = 0.4
RISK_AVERSION = 2.0

if SHARPE_RATIO <= 0 or RISK_AVERSION <= 0:
    raise ValueError("SHARPE_RATIO and RISK_AVERSION must be positive.")

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

In [ ]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "Results").is_dir() and (candidate / "Data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate a project root containing Results/ and Data/.")


def natural_key(name: str):
    return [int(part) if part.isdigit() else part.lower() for part in re.split(r"(\d+)", name)]


def discover_models(results_dir: Path, models=None, include_robustness=False):
    """Every Results/<name>/<name>.csv. Robustness runs (names ending in
    _refit<N>m, _aic<N>y, _resel<N>m or _seed<N>) are excluded unless
    requested or listed explicitly."""
    found = {}
    for folder in results_dir.iterdir():
        main_file = folder / f"{folder.name}.csv"
        if folder.is_dir() and main_file.is_file():
            found[folder.name] = main_file
    if models is not None:
        missing = [m for m in models if m not in found]
        if missing:
            print(f"Not yet run (skipped): {missing}")
        found = {m: found[m] for m in models if m in found}
    elif not include_robustness:
        found = {k: v for k, v in found.items()
                 if not re.search(r"_(refit\d+m|aic\d+y|resel\d+m|seed\d+)", k)}
    return dict(sorted(found.items(), key=lambda item: natural_key(item[0])))


def parse_source_dates(values):
    """Same date parser as NN/NN.ipynb."""
    text = values.astype(str).str.strip()
    slash_mask = text.str.contains("/", regex=False)
    parsed = pd.Series(pd.NaT, index=text.index, dtype="datetime64[ns]")
    parsed.loc[~slash_mask] = pd.to_datetime(text.loc[~slash_mask], format="%Y-%m-%d", errors="raise")
    parsed.loc[slash_mask] = pd.to_datetime(text.loc[slash_mask], dayfirst=True, errors="raise")
    return parsed


def load_prediction_file(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    if "date" not in frame.columns:
        raise ValueError(f"Missing date column in {path}")
    frame["date"] = pd.to_datetime(frame["date"], errors="raise")
    if frame["date"].duplicated().any():
        raise ValueError(f"Duplicate dates in {path}")
    frame = frame.set_index("date").sort_index()
    return frame.apply(pd.to_numeric, errors="coerce")


def load_actuals(tickers, data_dir: Path) -> pd.DataFrame:
    """Realised volatility, constructed exactly as in the model notebooks:
    same estimate type, same date parser, same duplicate policy."""
    series = []
    for ticker in tickers:
        path = data_dir / f"{ticker}.csv"
        if not path.is_file():
            raise FileNotFoundError(f"No realised-volatility file for {ticker}: {path}")
        frame = pd.read_csv(path)
        frame = frame.loc[frame["Type"].eq(ESTIMATE_TYPE), ["Date", VALUE_COLUMN]].copy()
        if frame.empty:
            raise ValueError(f"No {ESTIMATE_TYPE!r} observations in {path}")
        frame["Date"] = parse_source_dates(frame["Date"])
        duplicated = frame["Date"].duplicated(keep=False)
        if duplicated.any():
            dates = sorted(frame.loc[duplicated, "Date"].dt.date.unique())
            if DUPLICATE_POLICY == "drop":
                frame = frame.loc[~duplicated]
                action = "treated as missing"
            elif DUPLICATE_POLICY == "first":
                frame = frame.drop_duplicates("Date", keep="first")
                action = "first observation kept"
            else:
                raise ValueError("DUPLICATE_POLICY must be 'drop' or 'first'")
            print(f"Data note - {ticker}: repeated observations on {len(dates)} date(s) {action}: {dates}")
        item = pd.to_numeric(frame[VALUE_COLUMN], errors="coerce")
        item.index = frame["Date"]
        item.name = ticker
        series.append(item)
    return pd.concat(series, axis=1, sort=True)


ROOT = find_project_root(Path.cwd())
RESULTS_DIR = ROOT / "Results"
DATA_DIR = ROOT / "Data"

model_files = discover_models(RESULTS_DIR, MODELS, INCLUDE_ROBUSTNESS)
if not model_files:
    raise ValueError("No model result folders were found.")

print(f"Project root: {ROOT}")
print(f"Discovered {len(model_files)} models: {', '.join(model_files)}")

In [ ]:
predictions = {name: load_prediction_file(path) for name, path in model_files.items()}
models = list(predictions)

common_tickers = predictions[models[0]].columns
common_dates = predictions[models[0]].index
for model in models[1:]:
    common_tickers = common_tickers.intersection(predictions[model].columns)
    common_dates = common_dates.intersection(predictions[model].index)
common_tickers = common_tickers.sort_values()
actuals = load_actuals(common_tickers, DATA_DIR)
common_dates = common_dates.intersection(actuals.index).sort_values()

actual_array = actuals.loc[common_dates, common_tickers].to_numpy(dtype=float)
prediction_arrays = {
    model: predictions[model].loc[common_dates, common_tickers].to_numpy(dtype=float)
    for model in models
}

common_valid = np.isfinite(actual_array)
for predicted in prediction_arrays.values():
    common_valid &= np.isfinite(predicted)
# Equation (21) needs strictly positive volatility or variance under either input convention.
common_valid &= actual_array > 0
for predicted in prediction_arrays.values():
    common_valid &= predicted > 0

assets_per_date = common_valid.sum(axis=1)
usable_dates = assets_per_date > 0
if not usable_dates.any():
    raise ValueError("No common valid asset-date observations across the discovered models.")

sample_summary = pd.Series({
    "models": len(models),
    "common_tickers": len(common_tickers),
    "common_asset_date_observations": int(common_valid.sum()),
    "usable_dates": int(usable_dates.sum()),
    "first_usable_date": common_dates[usable_dates].min(),
    "last_usable_date": common_dates[usable_dates].max(),
    "mean_assets_per_usable_date": float(assets_per_date[usable_dates].mean()),
}, name="common sample")
display(sample_summary.to_frame())

In [ ]:
def realised_utility(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    if VALUES_ARE_VOLATILITY:
        volatility_ratio = actual / predicted
    else:
        if np.any(actual <= 0) or np.any(predicted <= 0):
            raise ValueError("Variance inputs must be strictly positive.")
        volatility_ratio = np.sqrt(actual / predicted)
    return (
        (SHARPE_RATIO**2 / RISK_AVERSION) * volatility_ratio
        - (SHARPE_RATIO**2 / (2 * RISK_AVERSION)) * volatility_ratio**2
    )


perfect_ru = SHARPE_RATIO**2 / (2 * RISK_AVERSION)
daily_utility = {}
summary_rows = []

for model in models:
    predicted = prediction_arrays[model]
    utility = np.full(actual_array.shape, np.nan)
    utility[common_valid] = realised_utility(actual_array[common_valid], predicted[common_valid])
    date_utility = np.divide(
        np.nansum(utility, axis=1),
        assets_per_date,
        out=np.full(len(common_dates), np.nan),
        where=assets_per_date > 0,
    )
    daily_utility[model] = pd.Series(date_utility, index=common_dates, name=model).dropna()
    mean_ru = float(np.nanmean(date_utility))
    summary_rows.append({
        "model": model,
        "realised_utility": mean_ru,
        "RU (% wealth)": mean_ru * 100,
        "perfect RU (% wealth)": perfect_ru * 100,
        "shortfall from perfect (bp)": (perfect_ru - mean_ru) * 10_000,
        "share of perfect utility (%)": mean_ru / perfect_ru * 100,
        "asset-date observations": int(common_valid.sum()),
        "dates": int(np.isfinite(date_utility).sum()),
    })

utility_summary = (
    pd.DataFrame(summary_rows)
    .sort_values("realised_utility", ascending=False)
    .reset_index(drop=True)
)

In [ ]:
display_columns = [
    "model", "RU (% wealth)", "perfect RU (% wealth)",
    "shortfall from perfect (bp)", "share of perfect utility (%)",
    "asset-date observations", "dates",
]
summary_style = (
    utility_summary[display_columns].style
    .format({
        "RU (% wealth)": "{:.6f}",
        "perfect RU (% wealth)": "{:.4f}",
        "shortfall from perfect (bp)": "{:.4f}",
        "share of perfect utility (%)": "{:.4f}",
        "asset-date observations": "{:,}",
        "dates": "{:,}",
    })
    .hide(axis="index")
    .set_caption("Realised utility ranking - higher is better")
    .set_table_styles([
        {"selector": "caption", "props": [("font-weight", "bold"), ("font-size", "14px"), ("margin-bottom", "8px")]},
        {"selector": "th", "props": [("background-color", "#16324F"), ("color", "white"), ("padding", "7px 10px")]},
        {"selector": "td", "props": [("padding", "6px 10px"), ("border-bottom", "1px solid #D9E2EC")]},
    ])
)
display(summary_style)
print("RU-TC not calculated: rolling 90-trading-day full bid-ask spreads are not available.")

In [ ]:
# Row model minus column model, expressed in basis points of wealth. Positive means the row model is better.
ru_by_model = utility_summary.set_index("model")["realised_utility"].reindex(models)
utility_gain_bp = pd.DataFrame(index=models, columns=models, dtype=float)
for row_model in models:
    for column_model in models:
        utility_gain_bp.loc[row_model, column_model] = (
            ru_by_model[row_model] - ru_by_model[column_model]
        ) * 10_000

gain_style = (
    utility_gain_bp.style
    .format("{:+.4f}")
    .set_caption("Pairwise realised-utility gain (basis points of wealth)")
    .set_table_styles([
        {"selector": "caption", "props": [("font-weight", "bold"), ("font-size", "14px"), ("margin-bottom", "8px")]},
        {"selector": "th", "props": [("background-color", "#16324F"), ("color", "white"), ("padding", "8px 12px")]},
        {"selector": "td", "props": [("text-align", "center"), ("padding", "8px 12px"), ("border-bottom", "1px solid #D9E2EC")]},
    ])
)
display(gain_style)
print("Positive entries favour the row model; negative entries favour the column model.")